# Demo 4 · The red pull request

**Slides:** "Test results, not exit codes" (slide 16), demo step 04.

A golden-result test trains a small seeded model and compares the accuracy with a committed value.
Here we make a change that looks harmless, a "tidier" preprocessing step, and watch the test catch it.
On GitHub the same test runs in `.github/workflows/golden.yml` on every pull request.

Runtime: CPU. Run the setup cells below first: about 2 minutes plus one session restart.

## 0 · Setup (run once per session)

Each Colab notebook has its own session, so run these cells even if you ran notebook 01. Restart the session when the install cell says so, then continue from the cell after it.

In [ ]:
REPO_URL = "https://github.com/keila-moral/reproducible-ml-demo"
import os
if not os.path.exists("/content/repo"):
    !git clone -q {REPO_URL} /content/repo
else:
    !git -C /content/repo pull -q   # already cloned in this session: get the latest version
%cd /content/repo
!git log --oneline -1

In [ ]:
# Install exactly what uv.lock records, verified by sha256 hash.
!pip install -q uv
!uv export --frozen --no-dev --no-emit-project --format requirements-txt -o requirements.lock.txt
!uv pip install --system -q --require-hashes -r requirements.lock.txt
!uv pip install --system -q --no-deps .
print("Done. Now restart the session: Runtime > Restart session, then continue below.")

In [ ]:
%cd /content/repo
import os, sys
os.environ["KERAS_BACKEND"] = "jax"
# Make the demo code importable here and in commands such as !dvc repro and !pytest.
os.environ["PYTHONPATH"] = "/content/repo/src"
sys.path.insert(0, "/content/repo/src")

## 1 · The golden value and the test

In [ ]:
!cat tests/golden/golden.json
!cat tests/golden/test_result.py

## 2 · On the main branch: green

In [ ]:
!dvc repro -q
!python -m pytest tests/golden -q

## 3 · A harmless-looking refactor

Replace the fixed `/ 16` scaling with per-batch standardisation. No error, no warning, and a reviewer might approve it.

In [ ]:
from pathlib import Path
p = Path("src/repro_demo/train.py")
src = p.read_text()
p.write_text(src.replace(
    'return features.astype("float32") / 16.0',
    'features = features.astype("float32")\n    return (features - features.mean()) / features.std()',
))
!git diff src/

## 4 · The test catches it: red

In [ ]:
!python -m pytest tests/golden -q 2>&1 | tail -6

Look at the numbers: the new accuracy is **higher** than the golden value, and the test still fails.
That is the point. An unexplained change in either direction needs a decision. If the change is wanted,
regenerate the value with `python -m repro_demo.make_golden` and commit it in the same pull request,
so the reviewer sees the result change next to the code that caused it.

## 5 · Revert: green again

In [ ]:
!git checkout -- src/
!python -m pytest tests/golden -q

## What to say

The code ran in both cases. Only a test on the **result** noticed that the number moved.
In CI this runs whenever `uv.lock`, the source, or the data definition changes, including
dependency updates opened by Dependabot.